# Research 32 — Three-seed VIP × region OOF average

**Hypothesis:** a third fold partition may further stabilize the VIP × region model. Compare a fixed 50/50 average of seeds 42 and 2025 with a fixed 1/3 average of seeds 42, 2025, and 1337, both at a fixed 20% contribution to the base. All component predictions are OOF for every row; this synthesis uses training labels only and is not independent validation. No fitting or weight search.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    values = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(values).all() and (values >= 0).all(), filename
    return values / values.sum(axis=1, keepdims=True)


p_base = load_probs("segment-oof.csv")
p42 = load_probs("segment-vip-region-balanced-oof.csv")
p2025 = load_probs("segment-vip-region-seed-2025-oof.csv")
p1337 = load_probs("segment-vip-region-seed-1337-oof.csv")
p_two = (p42 + p2025) / 2
p_three = (p42 + p2025 + p1337) / 3
candidates = {
    "Current reference": p_base,
    "80/20 seed 42 only": 0.8 * p_base + 0.2 * p42,
    "80/20 seed 2025 only": 0.8 * p_base + 0.2 * p2025,
    "80/20 two-seed average": 0.8 * p_base + 0.2 * p_two,
    "80/20 three-seed average": 0.8 * p_base + 0.2 * p_three,
}
fold_sets = {
    "seed42": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y)
    ),
    "seed2025": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=2025).split(np.zeros(len(y)), y)
    ),
    "seed1337": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=1337).split(np.zeros(len(y)), y)
    ),
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for seed_name, folds in fold_sets.items():
        for fold, (_, idx) in enumerate(folds, start=1):
            row[f"{seed_name}_fold_{fold}_macro_f1"] = f1_score(y[idx], pred[idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-vip-region-three-seed-average.csv", index=False)
pd.DataFrame(
    candidates["80/20 three-seed average"], columns=[f"prob_{i}" for i in range(4)]
).assign(segment=y).to_csv(
    ROOT / "predictions/segment-vip-region-three-seed-average-oof.csv", index=False
)
print("Three-seed OOF synthesis complete; Test not loaded.")

,candidate,macro_f1,weighted_f1,seed42_fold_1_macro_f1,seed42_fold_2_macro_f1,seed42_fold_3_macro_f1,seed2025_fold_1_macro_f1,seed2025_fold_2_macro_f1,seed2025_fold_3_macro_f1,seed1337_fold_1_macro_f1,seed1337_fold_2_macro_f1,seed1337_fold_3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.750797,0.749412,0.752462,0.749956,0.753540,0.749163
1,80/20 seed 42 only,0.751552,0.765087,0.750030,0.750601,0.753998,0.750568,0.750774,0.753298,0.750430,0.753978,0.750236
2,80/20 seed 2025 only,0.751547,0.765136,0.750114,0.750342,0.754156,0.751659,0.750233,0.752741,0.751252,0.754233,0.749139
3,80/20 two-seed average,0.751706,0.765361,0.749909,0.750991,0.754192,0.751228,0.750859,0.753021,0.751420,0.754147,0.749538
4,80/20 three-seed average,0.751862,0.765462,0.750224,0.750851,0.754482,0.751465,0.750765,0.753346,0.751549,0.754312,0.749708


Three-seed OOF synthesis complete; Test not loaded.
